# 証券営業インテリジェンス ハンズオン
## Part 4: Cortex Search（セマンティック検索）

このノートブックでは、**Cortex Search** を使ってニュース・商品説明書・アナリストレポートを意味的に検索します。

### Cortex Search とは

従来のキーワード検索と異なり、**意味（セマンティクス）を理解した検索**ができます。

| 比較項目 | 従来のキーワード検索 | Cortex Search |
|---|---|---|
| 検索方式 | 完全一致・部分一致 | ベクトル（意味）検索 |
| 日本語の揺らぎ | 「売却」と「手放す」は別 | 同じ意味として検索 |
| 複数キーワード | AND/OR が必要 | 自然文で検索可能 |
| 導入コスト | 低 | SQL一行で作成可 |

### 🚀 アハ体験ポイント

> **「証券担保ローン」と検索せず、「株を売らずにお金を借りる方法」で検索しても同じ商品説明書が出てくる！**
> 
> 顧客からの相談内容をそのまま検索クエリにできる

In [ ]:
%%sql -r result_use
USE DATABASE SNOWFINANCE_DB;
USE SCHEMA DEMO_SCHEMA;
USE WAREHOUSE COMPUTE_WH;
SELECT 'Ready!' AS STATUS;

## Step 1: Cortex Search Service の作成

3種類のデータに対してそれぞれ Cortex Search Service を作成します。

**作成するサービス:**
1. `NEWS_SEARCH_SERVICE` - マーケットニュース（50件）
2. `LOAN_DOCS_SEARCH_SERVICE` - ローン商品説明書（15件）
3. `ANALYST_REPORT_SEARCH_SERVICE` - アナリストレポート（30件）

> **Note**: インデックス構築には数分かかる場合があります。

In [ ]:
%%sql -r result_cs1
-- ============================================================
-- Cortex Search Service 1: マーケットニュース
-- ============================================================
CREATE OR REPLACE CORTEX SEARCH SERVICE NEWS_SEARCH_SERVICE
ON CONTENT
ATTRIBUTES CATEGORY, TITLE, RELATED_SECURITIES, PUBLISH_DATE, SENTIMENT
WAREHOUSE = COMPUTE_WH
TARGET_LAG = '1 day'
AS (
    SELECT
        NEWS_ID,
        PUBLISH_DATE,
        CATEGORY,
        TITLE,
        CONTENT,
        SUMMARY,
        RELATED_SECURITIES,
        SENTIMENT,
        IMPORTANCE
    FROM NEWS_ARTICLES
);

SELECT 'NEWS_SEARCH_SERVICE created!' AS STATUS;

In [ ]:
%%sql -r result_cs2
-- ============================================================
-- Cortex Search Service 2: ローン商品説明書
-- ============================================================
CREATE OR REPLACE CORTEX SEARCH SERVICE LOAN_DOCS_SEARCH_SERVICE
ON CONTENT
ATTRIBUTES PRODUCT_ID, DOC_TYPE, SECTION, TITLE
WAREHOUSE = COMPUTE_WH
TARGET_LAG = '1 day'
AS (
    SELECT
        DOC_ID,
        PRODUCT_ID,
        DOC_TYPE,
        SECTION,
        TITLE,
        CONTENT
    FROM LOAN_PRODUCT_DOCS
);

SELECT 'LOAN_DOCS_SEARCH_SERVICE created!' AS STATUS;

In [ ]:
%%sql -r result_cs3
-- ============================================================
-- Cortex Search Service 3: アナリストレポート
-- ============================================================
CREATE OR REPLACE CORTEX SEARCH SERVICE ANALYST_REPORT_SEARCH_SERVICE
ON CONTENT
ATTRIBUTES SECURITY_CODE, SECURITY_NAME, RATING, PUBLISH_DATE, ANALYST_NAME
WAREHOUSE = COMPUTE_WH
TARGET_LAG = '1 day'
AS (
    SELECT
        REPORT_ID,
        PUBLISH_DATE,
        SECURITY_CODE,
        SECURITY_NAME,
        ANALYST_NAME,
        RATING,
        TARGET_PRICE,
        REPORT_TITLE,
        EXECUTIVE_SUMMARY,
        INVESTMENT_THESIS AS CONTENT,
        KEY_RISKS
    FROM ANALYST_REPORTS
);

SELECT 'ANALYST_REPORT_SEARCH_SERVICE created!' AS STATUS;

In [ ]:
%%sql -r result_verify_cs
SHOW CORTEX SEARCH SERVICES IN SCHEMA DEMO_SCHEMA;

## Step 2: セマンティック検索のデモ

### 2-1. ローン商品説明書の意味検索

**通常の検索**: 「証券担保ローン」と入力しないとヒットしない

**Cortex Search**: 「株を売らずに資金を調達したい」でもヒットする！

In [ ]:
%%sql -r result_search1
-- ============================================================
-- 意味検索デモ: 「株を売らずに資金を調達する方法」
-- ============================================================
-- SEARCH_PREVIEW を使って Cortex Search に問い合わせ
SELECT
    ROUND(f.value:score::FLOAT, 3) AS 類似度スコア,
    f.value:DOC_ID::STRING         AS ドキュメントID,
    f.value:TITLE::STRING          AS タイトル,
    f.value:SECTION::STRING        AS セクション,
    LEFT(f.value:CONTENT::STRING, 150) AS 内容（先頭150字）
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'LOAN_DOCS_SEARCH_SERVICE',
        '{
            "query": "株を売らずに資金を調達したい",
            "columns": ["DOC_ID", "TITLE", "SECTION", "CONTENT"],
            "limit": 5
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

In [ ]:
%%sql -r result_search2
-- ============================================================
-- 意味検索デモ: 「孫への教育費用を非課税で渡す方法」
-- ============================================================
SELECT
    ROUND(f.value:score::FLOAT, 3) AS 類似度スコア,
    f.value:TITLE::STRING          AS タイトル,
    f.value:SECTION::STRING        AS セクション,
    LEFT(f.value:CONTENT::STRING, 200) AS 内容（先頭200字）
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'LOAN_DOCS_SEARCH_SERVICE',
        '{
            "query": "孫への教育費用を非課税で渡す方法",
            "columns": ["TITLE", "SECTION", "CONTENT"],
            "limit": 3
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

### 2-2. ニュース検索（フィルタ付き）

特定の条件でフィルタリングしながらセマンティック検索ができます。

**例**: 「トヨタ」関連のポジティブニュースのみを検索

In [ ]:
%%sql -r result_search3
-- トヨタ関連のポジティブニュースを検索
SELECT
    ROUND(f.value:score::FLOAT, 3)    AS 類似度スコア,
    f.value:PUBLISH_DATE::STRING      AS 発行日,
    f.value:TITLE::STRING             AS タイトル,
    f.value:SENTIMENT::STRING         AS センチメント,
    LEFT(f.value:CONTENT::STRING, 150) AS 内容（先頭150字）
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'NEWS_SEARCH_SERVICE',
        '{
            "query": "トヨタ自動車の業績と株価見通し",
            "columns": ["PUBLISH_DATE", "TITLE", "CONTENT", "SENTIMENT"],
            "filter": {"@eq": {"SENTIMENT": "ポジティブ"}},
            "limit": 5
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

### 2-3. アナリストレポート検索

顧客が保有している銘柄（トヨタ）の最新アナリスト評価を検索します。

In [ ]:
%%sql -r result_search4
-- トヨタのアナリストレポートを検索
SELECT
    ROUND(f.value:score::FLOAT, 3)          AS 類似度スコア,
    f.value:PUBLISH_DATE::STRING            AS 発行日,
    f.value:SECURITY_NAME::STRING           AS 銘柄名,
    f.value:ANALYST_NAME::STRING            AS アナリスト名,
    f.value:RATING::STRING                  AS 投資判断,
    f.value:TARGET_PRICE::STRING            AS 目標株価,
    LEFT(f.value:CONTENT::STRING, 200)      AS 投資根拠（先頭200字）
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'ANALYST_REPORT_SEARCH_SERVICE',
        '{
            "query": "トヨタ自動車 ハイブリッド EV 業績見通し",
            "columns": ["PUBLISH_DATE", "SECURITY_NAME", "ANALYST_NAME", "RATING", "TARGET_PRICE", "CONTENT"],
            "filter": {"@eq": {"SECURITY_CODE": "7203"}},
            "limit": 3
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

## Step 3: 山田様の相談への活用（総合デモ）

「トヨタ株を売って孫の留学費用を捻出したい」という相談に対して、
Cortex Search で関連情報を一括収集します。

In [ ]:
%%sql -r result_yamada_search
-- ============================================================
-- 山田様（C001）の相談に関連する情報を全サービスから収集
-- ============================================================

-- 1. 関連ニュース（税制・トヨタ）
SELECT '【ニュース】' AS 種別,
    f.value:PUBLISH_DATE::STRING AS 日付,
    f.value:TITLE::STRING AS タイトル,
    f.value:SENTIMENT::STRING AS センチメント,
    ROUND(f.value:score::FLOAT, 3) AS スコア
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'NEWS_SEARCH_SERVICE',
        '{"query":"トヨタ株売却 税金 教育資金 相続","columns":["PUBLISH_DATE","TITLE","SENTIMENT"],"limit":3}'
    ) AS r
), LATERAL FLATTEN(input => PARSE_JSON(r):results) AS f

UNION ALL

-- 2. 関連商品説明書（証券担保ローン・教育信託）
SELECT '【商品説明書】',
    NULL,
    f.value:TITLE::STRING,
    f.value:SECTION::STRING,
    ROUND(f.value:score::FLOAT, 3)
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'LOAN_DOCS_SEARCH_SERVICE',
        '{"query":"株売却 節税 教育資金 2000万円 調達","columns":["TITLE","SECTION"],"limit":3}'
    ) AS r
), LATERAL FLATTEN(input => PARSE_JSON(r):results) AS f

ORDER BY スコア DESC;

## まとめ

Part 4 完了！Cortex Search が動作することを確認しました。

### 作成した Cortex Search Services

| サービス名 | 対象データ | 件数 |
|---|---|---|
| NEWS_SEARCH_SERVICE | マーケットニュース | 50件 |
| LOAN_DOCS_SEARCH_SERVICE | ローン商品説明書 | 15件 |
| ANALYST_REPORT_SEARCH_SERVICE | アナリストレポート | 30件 |

### Cortex Search の強み

- **意味理解**: 「株を売らずに資金調達」→「証券担保ローン」を発見
- **フィルタ**: 感情（ポジティブ/ネガティブ）、銘柄コードなどで絞込み
- **スコアリング**: 関連度スコアで上位結果を表示
- **リアルタイム更新**: TARGET_LAG で自動的にインデックスを更新

**次のステップ:** `05_cortex_agent.ipynb` で Cortex Agent を作成してください。